In [1]:
import pandas as pd
import requests

In [2]:
url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=20000"

In [4]:
response = requests.get(url)
response.status_code

200

In [5]:
data = response.json()

In [7]:
population = pd.DataFrame(data[1])
population.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,788844284.0,,,0
1,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,769280888.0,,,0
2,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,750491370.0,,,0
3,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,731821393.0,,,0
4,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,713090928.0,,,0


In [8]:
population = population.rename(columns={
    "country": "Country",
    "countryiso3code": "Country_Code",
    "date": "Year",
    "value": "Population"
})

In [14]:
population.head()

,indicator,Country,Country_Code,Year,Population,unit,obs_status,decimal
0,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...",Africa Eastern and Southern,AFE,2025,788844284.0,,,0
1,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...",Africa Eastern and Southern,AFE,2024,769280888.0,,,0
2,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...",Africa Eastern and Southern,AFE,2023,750491370.0,,,0
3,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...",Africa Eastern and Southern,AFE,2022,731821393.0,,,0
4,"{'id': 'SP.POP.TOTL', 'value': 'Population, to...",Africa Eastern and Southern,AFE,2021,713090928.0,,,0


In [18]:
population["Year"].dtype
population["Year"] = pd.to_numeric(population["Year"])

In [19]:
population["Population"].isnull().sum()

np.int64(96)

In [20]:
population_clean = population.dropna(
    subset=["Population"]
).copy()

In [21]:
population_clean.shape

(17394, 8)

In [22]:
country_url = "https://api.worldbank.org/v2/country?format=json&per_page=400"

In [23]:
country_response = requests.get(country_url)

In [24]:
country_data = country_response.json()

In [25]:
countries = pd.DataFrame(country_data[1])

In [26]:
countries["Region"] = countries["region"].apply(
    lambda x: x["value"]
)

In [27]:
countries_only = countries[
    countries["Region"] != "Aggregates"
].copy()

In [28]:
population_clean = population_clean.merge(
    countries_only[["id", "Region"]],
    left_on="Country_Code",
    right_on="id",
    how="inner"
)

In [33]:
population_analysis = population_clean[
    ["Country", "Country_Code", "Year", "Population", "Region"]
].copy()

In [34]:
population_analysis.head()

,Country,Country_Code,Year,Population,Region
0,Afghanistan,AFG,2025,43844111.0,"Middle East, North Africa, Afghanistan & Pakistan"
1,Afghanistan,AFG,2024,42647492.0,"Middle East, North Africa, Afghanistan & Pakistan"
2,Afghanistan,AFG,2023,41454761.0,"Middle East, North Africa, Afghanistan & Pakistan"
3,Afghanistan,AFG,2022,40578842.0,"Middle East, North Africa, Afghanistan & Pakistan"
4,Afghanistan,AFG,2021,40000412.0,"Middle East, North Africa, Afghanistan & Pakistan"


In [35]:
population_analysis.duplicated(
    subset=["Country_Code", "Year"]
).sum()

np.int64(0)

In [36]:
population_analysis[
    population_analysis["Country"] == "India"
][
    ["Country", "Year", "Population", "Region"]
]

,Country,Year,Population,Region
5874,India,2025,1.463866e+09,South Asia
5875,India,2024,1.450936e+09,South Asia
5876,India,2023,1.438070e+09,South Asia
5877,India,2022,1.425423e+09,South Asia
5878,India,2021,1.414204e+09,South Asia
...,...,...,...,...
5935,India,1964,4.792296e+08,South Asia
5936,India,1963,4.681386e+08,South Asia
5937,India,1962,4.572831e+08,South Asia
5938,India,1961,4.465647e+08,South Asia


In [37]:
population_analysis.to_csv(
    "../data/population_clean.csv",
    index=False
)